In [ ]:
from google.colab import files

In [ ]:
upload = files.upload()
filename = next(iter(upload))

In [ ]:
import pandas as pd
import numpy as np
import zipfile

with zipfile.ZipFile("olist_customers_dataset.csv.zip", "r") as z:
  print(z.namelist())
  z.extractall("extracted_data")

In [ ]:
df = pd.read_csv("extracted_data/olist_customers_dataset.csv")


In [ ]:
upload = files.upload()
filename = next(iter(upload))

In [ ]:
with zipfile.ZipFile("olist_order_payments_dataset.csv.zip", "r") as z:
  print(z.namelist())
  z.extractall("extracted_data")

In [ ]:
op = pd.read_csv("extracted_data/olist_order_payments_dataset.csv")


In [ ]:
upload = files.upload()
filename = next(iter(upload))

In [ ]:
with zipfile.ZipFile("olist_order_items_dataset.csv.zip", "r") as z:
  print(z.namelist())
  z.extractall("extracted_data")

In [ ]:
upload = files.upload()
filename = next(iter(upload))

In [ ]:
with zipfile.ZipFile("olist_orders_dataset.csv.zip", "r") as z:
  print(z.namelist())
  z.extractall("extracted_data")

In [ ]:
oo = pd.read_csv("extracted_data/olist_orders_dataset.csv")

In [ ]:
oi = pd.read_csv("extracted_data/olist_order_items_dataset.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
op.head()

In [ ]:
op.tail()

In [ ]:
op.describe()

In [ ]:
op.dtypes

In [ ]:
oi.head()

In [ ]:
oi.tail()

In [ ]:
oi.describe()

In [ ]:
oi.dtypes

In [ ]:
oo.head()

In [ ]:
oo.tail()

In [ ]:
oo.dtypes

In [ ]:
oi.dtypes

In [ ]:
df.head()

In [ ]:
revenue = np.sum(oi["price"])

In [ ]:
revenue

In [ ]:
op["payment_value"].unique()

In [ ]:
revenue

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
oi.info()

In [ ]:
oi.shape

In [ ]:
oi.columns

In [ ]:
oo.shape

In [ ]:
oo.columns

In [ ]:
oo.info()

In [ ]:
op.info()

In [ ]:
op.shape

In [ ]:
op.columns

In [ ]:
merged_df = pd.merge(df, oo, on="customer_id")
merged_df

In [ ]:
merged_df.shape


In [ ]:
merged_df[merged_df["order_status"].isna()]

In [ ]:
merged_df["customer_state"].unique()

In [ ]:
merge_ordered = pd.merge(oo, oi, on="order_id")
merge_ordered

In [ ]:
merge_ordered.isna().sum()

In [ ]:
delay = pd.to_datetime(merged_df["order_delivered_customer_date"]).dt.day-pd.to_datetime(merged_df["order_estimated_delivery_date"]).dt.day
delay

In [ ]:
delay.describe()

In [ ]:
cancelled_df = merged_df[merged_df["order_status"] == "canceled"]
cancelled_df

In [ ]:
final_merge = pd.merge(merged_df,op, on="order_id")
final_merge

In [ ]:
total_value = final_merge["payment_value"].sum()
total_value

In [ ]:
delayed_df = (delay>0).sum()
delayed_df

In [ ]:
canceled_value = final_merge.loc[final_merge["order_status"] == "canceled", "payment_value"].sum()
total_revenue = total_value - canceled_value
total_revenue

In [ ]:
merged = pd.merge(final_merge, oi, on="order_id")
merged

In [ ]:
merged.describe()

In [ ]:
mean_value = merged["payment_value"].mean()
mean_value

In [ ]:
p33, p66 = merged["payment_value"].quantile([0.33,0.66])
merged["Performance_group"]= np.where(merged["payment_value"] > p66 ,"High", np.where(merged["payment_value"]>p33, "Medium", "low"))

In [ ]:
merged["Performance_group"]

In [ ]:
counts = merged["Performance_group"].value_counts()
counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.pie(counts, labels=counts.index, autopct='%1.1f%%')
plt.title('Performance Group Distribution')
plt.show()


In [ ]:
plt.bar(counts.index, counts.values)
plt.xlabel('Performance Group')
plt.ylabel('Count')
plt.title('Performance Group Distribution')
plt.yscale("log")
plt.show()

In [ ]:
stats = merged.groupby("customer_state")["payment_value"].sum()
stats.reset_index()
stats_sorted = stats.sort_values(ascending=False)
stats_sorted.reset_index()

In [ ]:
plt.bar(stats_sorted.index, stats_sorted.values)
plt.xlabel('State')
plt.ylabel('Revenue')
plt.title('Revenue by State')
plt.show()

In [ ]:
stats_norm = (stats - stats.min()) / (stats.max() - stats.min())
stats_df = stats.to_frame(name="payment_value")
stats_df["score_norm"] = stats_norm
stats_df

In [ ]:
import matplotlib.pyplot as plt

# Calculate order volume per state
order_volume = merged.groupby('customer_state')['order_id'].nunique().reset_index(name='order_count')

# Create the scatter plot using the string column names from our grouped data
plt.scatter(order_volume['customer_state'], order_volume['order_count'])
plt.xlabel('Customer State')
plt.ylabel('Order Count')
plt.title('State by Order Volume')
plt.show()